0. Import dan Load Data

In [1]:
from datetime import date
from pathlib import Path
import csv
from collections import Counter

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / 'data' / 'raw').exists():
        PROJECT_ROOT = p
        break

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PATH = RAW_DIR / 'stocks.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'stocks_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} tidak ada. Isi folder: {list(RAW_DIR.iterdir())}')

# Pemisah file ini adalah ';'. Semua kolom dibaca sebagai teks supaya tidak ada nilai yang hilang diam-diam.
df = pl.read_csv(DATA_PATH, separator=';', infer_schema=False, truncate_ragged_lines=True)

print('Membaca:', DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print('Kolom:', df.columns)

WAJIB = {'datetime', 'symbol', 'open', 'high', 'low', 'close', 'volume'}
hilang = WAJIB - set(df.columns)
if hilang:
    raise ValueError(f'Kolom tidak ditemukan: {hilang}. Kolom yang ada: {df.columns}')
df.head()

Membaca: /home/jovyan/work/data/raw/stocks.csv
Ukuran awal: 1,048,575 baris x 7 kolom
Kolom: ['datetime', 'symbol', 'open', 'high', 'low', 'close', 'volume']


datetime,symbol,open,high,low,close,volume
str,str,str,str,str,str,str
"""30/06/2005 02:00""","""IDX:AALI""","""311192455""","""3131620275""","""311192455""","""3131620275""","""15809260"""
"""01/07/2005 02:00""","""IDX:AALI""","""311192455""","""311192455""","""30725331""","""3092228825""","""9012100"""
"""04/07/2005 02:00""","""IDX:AALI""","""30725331""","""30725331""","""29937502""","""303314165""","""32265880"""
"""05/07/2005 02:00""","""IDX:AALI""","""303314165""","""303314165""","""3013445925""","""3013445925""","""19287180"""
"""06/07/2005 02:00""","""IDX:AALI""","""303314165""","""30725331""","""303314165""","""3052837375""","""16323330"""


1. Profiling DuckDB + cek struktur baris

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)


def sql(query: str) -> pl.DataFrame:
    """Jalankan SQL DuckDB dan kembalikan Polars DataFrame (kompatibel lintas versi DuckDB)."""
    hasil = connection.execute(query)
    return hasil.pl() if hasattr(hasil, 'pl') else pl.from_arrow(hasil.to_arrow_table())


profil = sql('SUMMARIZE SELECT * FROM dataset')
display(profil)

# Struktur baris: setiap baris harus punya jumlah kolom sama dengan header
with open(DATA_PATH, 'r', encoding='utf-8', errors='replace', newline='') as f:
    reader = csv.reader(f, delimiter=';')
    n_header = len(next(reader))
    distribusi_kolom = Counter(len(row) for row in reader)
print('Jumlah kolom header:', n_header)
print('Distribusi jumlah kolom per baris:', dict(distribusi_kolom))
n_baris_struktur_salah = sum(v for k, v in distribusi_kolom.items() if k != n_header)
print('Baris dengan jumlah kolom tidak sesuai:', f'{n_baris_struktur_salah:,}')

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""datetime""","""VARCHAR""","""01/02/1991 02:00""","""31/12/2008 02:00""",7654,null,null,null,null,null,1048575,0.00
"""symbol""","""VARCHAR""","""IDX:AALI""","""IDX:MLBI""",340,null,null,null,null,null,1048575,0.00
"""open""","""VARCHAR""","""1,00016E+12""","""999999""",50980,null,null,null,null,null,1048575,0.00
"""high""","""VARCHAR""","""1,00016E+12""","""999999""",51413,null,null,null,null,null,1048575,0.00
"""low""","""VARCHAR""","""1,00253E+12""","""999999""",50766,null,null,null,null,null,1048575,0.00
"""close""","""VARCHAR""","""1,00016E+12""","""999999""",49906,null,null,null,null,null,1048575,0.00
"""volume""","""VARCHAR""","""0""","""99999000""",439269,null,null,null,null,null,1048575,0.00


Jumlah kolom header: 7
Distribusi jumlah kolom per baris: {7: 1048575}
Baris dengan jumlah kolom tidak sesuai: 0


2. Deteksi Missing Value

In [3]:
jumlah_baris = len(df)
missing_summary = pl.DataFrame({
    'kolom': df.columns,
    'jumlah_null': [df[c].null_count() for c in df.columns],
    'jumlah_kosong': [(df[c].str.strip_chars() == '').sum() for c in df.columns],
}).with_columns(
    ((pl.col('jumlah_null') + pl.col('jumlah_kosong')) / jumlah_baris * 100)
    .round(3).alias('persen_bermasalah')
)
display(missing_summary)

print('Contoh format teks mentah per kolom angka:')
display(df.select(['datetime', 'symbol', 'open', 'high', 'low', 'close', 'volume']).sample(8, seed=42))

kolom,jumlah_null,jumlah_kosong,persen_bermasalah
str,i64,i64,f64
"""datetime""",0,0,0.0
"""symbol""",0,0,0.0
"""open""",0,0,0.0
"""high""",0,0,0.0
"""low""",0,0,0.0
"""close""",0,0,0.0
"""volume""",0,0,0.0


Contoh format teks mentah per kolom angka:


datetime,symbol,open,high,low,close,volume
str,str,str,str,str,str,str
"""01/04/2014 02:00""","""IDX:KBRI""","""500""","""500""","""500""","""500""","""110000"""
"""14/03/2023 02:00""","""IDX:BNBR""","""610""","""610""","""570""","""570""","""2096367000"""
"""09/11/2023 02:00""","""IDX:MICE""","""5250""","""5250""","""5100""","""5250""","""31000"""
"""21/09/2015 02:00""","""IDX:INCO""","""16243788""","""168344712""","""1599767""","""165391296""","""1,34429E+15"""
"""19/04/2016 02:00""","""IDX:JTPE""","""565""","""570""","""5625""","""570""","""55924000"""
"""05/10/2022 02:00""","""IDX:GEMS""","""73000""","""75000""","""73000""","""74000""","""1509000"""
"""13/02/2006 02:00""","""IDX:ARTA""","""4300""","""4300""","""4300""","""4300""","""150000"""
"""29/01/2019 02:00""","""IDX:GLOB""","""4040""","""4800""","""3640""","""4040""","""226000"""


3. Normalisasi teks dan parsing tipe data

In [4]:
ANGKA = ['open', 'high', 'low', 'close', 'volume']
HARGA = ['open', 'high', 'low', 'close']


def ke_angka(kolom: str) -> pl.Expr:
    """Ubah teks jadi float. Kalau ada koma, format Indonesia: titik = ribuan, koma = desimal.
    Notasi ilmiah seperti '4,61722E+15' ikut terbaca. Yang gagal menjadi null."""
    s = pl.col(kolom).str.strip_chars().str.replace_all(r'\s', '')
    return (
        pl.when(s.str.contains(','))
        .then(s.str.replace_all(r'\.', '').str.replace(',', '.'))
        .otherwise(s)
        .cast(pl.Float64, strict=False)
    )


df_parsed_full = (
    df.with_columns(
        pl.col('symbol').str.strip_chars().str.to_uppercase()
          .str.replace(r'^IDX:', '').alias('ticker'),
        pl.col('volume').str.contains(r'(?i)e[+-]?\d+').fill_null(False)
          .alias('volume_notasi_ilmiah'),
        pl.col('datetime').str.strip_chars()
          .str.strptime(pl.Datetime, '%d/%m/%Y %H:%M', strict=False).alias('waktu'),
    )
    .with_columns([ke_angka(c).alias(f'{c}_num') for c in ANGKA])
)

# Laporan gagal parse: ada teks aslinya, tapi hasil parse null
gagal = {'datetime': (df_parsed_full['waktu'].is_null() & df['datetime'].is_not_null()).sum()}
for c in ANGKA:
    gagal[c] = (df_parsed_full[f'{c}_num'].is_null() & df[c].is_not_null()).sum()
display(pl.DataFrame({'kolom': list(gagal), 'gagal_parse': list(gagal.values())}))
print('Volume dalam notasi ilmiah (presisi sudah terpotong):',
      f"{df_parsed_full['volume_notasi_ilmiah'].sum():,}")

df_parsed = df_parsed_full.select(
    pl.col('waktu').dt.date().alias('tanggal'),
    'ticker',
    *[pl.col(f'{c}_num').alias(c) for c in ANGKA],
    'volume_notasi_ilmiah',
)

# Buang baris yang wajib-nya kosong / tidak terbaca / bukan angka hingga
n_sebelum = len(df_parsed)
df_missing_handled = df_parsed.filter(
    pl.col('tanggal').is_not_null()
    & pl.col('ticker').is_not_null() & (pl.col('ticker') != '')
    & pl.all_horizontal([pl.col(c).is_not_null() & pl.col(c).is_finite() for c in ANGKA])
)
print(f'Baris dibuang (kosong / gagal parse): {n_sebelum - len(df_missing_handled):,}')
df_missing_handled.head()

kolom,gagal_parse
str,i64
"""datetime""",0
"""open""",0
"""high""",0
"""low""",0
"""close""",0
"""volume""",0


Volume dalam notasi ilmiah (presisi sudah terpotong): 53,295
Baris dibuang (kosong / gagal parse): 0


tanggal,ticker,open,high,low,close,volume,volume_notasi_ilmiah
date,str,f64,f64,f64,f64,f64,bool
2005-06-30,"""AALI""",3.11192455e8,3.1316e9,3.11192455e8,3.1316e9,1.580926e7,false
2005-07-01,"""AALI""",3.11192455e8,3.11192455e8,3.0725331e7,3.0922e9,9.0121e6,false
2005-07-04,"""AALI""",3.0725331e7,3.0725331e7,2.9937502e7,3.03314165e8,3.226588e7,false
2005-07-05,"""AALI""",3.03314165e8,3.03314165e8,3.0134e9,3.0134e9,1.928718e7,false
2005-07-06,"""AALI""",3.03314165e8,3.0725331e7,3.03314165e8,3.0528e9,1.632333e7,false


4. Validitas Logika Harga

In [5]:
aturan = {
    'harga <= 0': pl.any_horizontal([pl.col(c) <= 0 for c in HARGA]),
    'volume < 0': pl.col('volume') < 0,
    'high < low': pl.col('high') < pl.col('low'),
    'open di luar [low, high]': (pl.col('open') > pl.col('high')) | (pl.col('open') < pl.col('low')),
    'close di luar [low, high]': (pl.col('close') > pl.col('high')) | (pl.col('close') < pl.col('low')),
    'tanggal di luar 1990 - hari ini': ~pl.col('tanggal').is_between(date(1990, 1, 1), date.today()),
}
laporan = pl.DataFrame({
    'aturan': list(aturan),
    'jumlah_baris': [df_missing_handled.filter(a).height for a in aturan.values()],
}).with_columns(
    (pl.col('jumlah_baris') / len(df_missing_handled) * 100).round(3).alias('persen')
)
display(laporan)

df_valid = df_missing_handled.filter(~pl.any_horizontal(list(aturan.values())))
n_logika_dibuang = len(df_missing_handled) - len(df_valid)
print(f'Baris dibuang karena melanggar logika harga (gabungan, tanpa hitung ganda): {n_logika_dibuang:,}')

# Diagnostik skala (hanya dilaporkan, tidak dibuang):
# dalam sehari, rentang open/high/low/close seharusnya kecil. Rasio > 5 menandakan
# titik desimal hilang atau skala angka tidak konsisten di dalam satu baris.
skala_aneh = df_valid.filter(
    (pl.max_horizontal(HARGA) / pl.min_horizontal(HARGA)) > 5
)
print(f'Baris dengan rasio max/min OHLC > 5 (curiga skala rusak): {len(skala_aneh):,} '
      f'({len(skala_aneh) / len(df_valid) * 100:.3f}%)')
display(skala_aneh.head(10))

# Panjang digit harga close: kalau sangat bervariasi, skala harga tidak konsisten
display(
    df_valid.with_columns(pl.col('close').cast(pl.Int64).cast(pl.Utf8).str.len_chars().alias('digit_close'))
    .group_by('digit_close').len().sort('digit_close')
)

aturan,jumlah_baris,persen
str,i64,f64
"""harga <= 0""",0,0.0
"""volume < 0""",0,0.0
"""high < low""",37876,3.612
"""open di luar [low, high]""",56953,5.431
"""close di luar [low, high]""",58792,5.607
"""tanggal di luar 1990 - hari in…",0,0.0


Baris dibuang karena melanggar logika harga (gabungan, tanpa hitung ganda): 71,317
Baris dengan rasio max/min OHLC > 5 (curiga skala rusak): 35,379 (3.620%)


tanggal,ticker,open,high,low,close,volume,volume_notasi_ilmiah
date,str,f64,f64,f64,f64,f64,bool
2005-06-30,"""AALI""",3.11192455e8,3.1316e9,3.11192455e8,3.1316e9,1.580926e7,false
2005-07-12,"""AALI""",3.19070745e8,3.2104e9,3.151316e6,3.1710e9,1.596793e7,false
2005-07-13,"""AALI""",3.151316e6,3.1710e9,3.151316e6,3.151316e6,1.806864e7,false
2005-07-15,"""AALI""",3.1316e9,3.1316e9,3.11192455e8,3.11192455e8,6.40367e6,false
2005-07-19,"""AALI""",3.1316e9,3.1316e9,3.11192455e8,3.11192455e8,526760.0,false
2005-07-22,"""AALI""",3.2300989e7,3.2892e9,3.2300989e7,3.2300989e7,2.14577e7,false
2005-07-26,"""AALI""",3.2300989e7,3.26949035e8,3.2300989e7,3.26949035e8,1.797979e7,false
2005-07-27,"""AALI""",3.2498e9,3.2892e9,3.2300989e7,3.2300989e7,2.265085e7,false
2005-07-29,"""AALI""",3.2300989e7,3.2498e9,3.2300989e7,3.2498e9,1.110012e7,false


digit_close,len
u32,u32
2,2755
3,141638
4,437544
5,208497
6,28298
…,…
9,60121
10,13558
11,12205


5. Deteksi dan penanganan outlier

In [6]:
FAKTOR_IQR = 3.0

# 5a. Outlier volume: IQR pada log10(volume), hanya untuk volume > 0
df_fitur = df_valid.with_columns(
    pl.when(pl.col('volume') > 0).then(pl.col('volume').log10()).alias('log_volume')
)
q1 = df_fitur['log_volume'].quantile(0.25)
q3 = df_fitur['log_volume'].quantile(0.75)
iqr = q3 - q1
batas_bawah_logvol = q1 - FAKTOR_IQR * iqr
batas_atas_logvol = q3 + FAKTOR_IQR * iqr
print(f'log10(volume): Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, '
      f'batas={batas_bawah_logvol:.2f} s/d {batas_atas_logvol:.2f} '
      f'(volume {10**batas_bawah_logvol:,.0f} s/d {10**batas_atas_logvol:,.0f})')

df_outlier_handled = (
    df_fitur
    .filter(
        pl.col('log_volume').is_null()   # volume 0 tetap lolos, dilaporkan terpisah
        | pl.col('log_volume').is_between(batas_bawah_logvol, batas_atas_logvol)
    )
    .drop('log_volume')
)
n_outlier = len(df_fitur) - len(df_outlier_handled)
print(f'Outlier volume dibuang: {n_outlier:,}')

# 5b. Lonjakan harga antar hari (hanya dilaporkan: stock split / reverse split bisa sah)
lompatan = (
    df_outlier_handled.sort(['ticker', 'tanggal'])
    .with_columns((pl.col('close') / pl.col('close').shift(1).over('ticker')).alias('rasio_close'))
    .filter((pl.col('rasio_close') > 5) | (pl.col('rasio_close') < 0.2))
)
print(f'Lonjakan close > 5x atau < 0.2x antar hari (hanya dilaporkan): {len(lompatan):,}')
display(lompatan.select(['tanggal', 'ticker', 'close', 'rasio_close']).head(10))

# 5c. Laporan lain (tidak dibuang)
n_volume_nol = df_outlier_handled.filter(pl.col('volume') == 0).height
n_akhir_pekan = df_outlier_handled.filter(pl.col('tanggal').dt.weekday() >= 6).height
print(f'Volume = 0 (hari tanpa transaksi, hanya dilaporkan): {n_volume_nol:,}')
print(f'Tanggal akhir pekan (hanya dilaporkan): {n_akhir_pekan:,}')

log10(volume): Q1=5.70, Q3=8.07, IQR=2.38, batas=-1.43 s/d 15.20 (volume 0 s/d 1,582,011,058,684,875)
Outlier volume dibuang: 11,719
Lonjakan close > 5x atau < 0.2x antar hari (hanya dilaporkan): 27,519


tanggal,ticker,close,rasio_close
date,str,f64,f64
2005-07-13,"""AALI""",3.151316e6,0.000994
2005-07-15,"""AALI""",3.11192455e8,98.75
2005-07-22,"""AALI""",3.2300989e7,0.103797
2005-07-26,"""AALI""",3.26949035e8,10.121951
2005-07-27,"""AALI""",3.2300989e7,0.098795
2005-07-29,"""AALI""",3.2498e9,100.609756
2005-08-10,"""AALI""",3.3088818e7,0.010182
2005-08-12,"""AALI""",3.2104e9,97.02381
2005-08-15,"""AALI""",3.151316e6,0.000982


Volume = 0 (hari tanpa transaksi, hanya dilaporkan): 18,114
Tanggal akhir pekan (hanya dilaporkan): 0


6. Deduplikasi

In [7]:
KUNCI = ['ticker', 'tanggal']

# 6a. Duplikat penuh (semua kolom sama)
df_beda = df_outlier_handled.unique(maintain_order=True)
n_duplikat_penuh = len(df_outlier_handled) - len(df_beda)

# 6b. Kunci (ticker, tanggal) sama tapi nilai berbeda: simpan yang terakhir, laporkan
konflik_df = (
    df_beda.group_by(KUNCI).len().filter(pl.col('len') > 1).drop('len')
)
n_konflik_kunci = konflik_df.height
print(f'Duplikat penuh dibuang: {n_duplikat_penuh:,}')
print(f'Kunci {KUNCI} dengan nilai berbeda: {n_konflik_kunci:,}')
if n_konflik_kunci:
    print('Contoh konflik untuk dicek manual:')
    display(df_beda.join(konflik_df, on=KUNCI, how='inner').sort(KUNCI).head(10))

df_dedup = (
    df_beda.unique(subset=KUNCI, keep='last', maintain_order=True)
    .sort(KUNCI)
)
n_konflik_dibuang = len(df_beda) - len(df_dedup)
print(f'Baris dibuang karena kunci ganda: {n_konflik_dibuang:,}')
print(f'Baris akhir: {len(df_dedup):,}')

Duplikat penuh dibuang: 0
Kunci ['ticker', 'tanggal'] dengan nilai berbeda: 0
Baris dibuang karena kunci ganda: 0
Baris akhir: 965,539


7. Validasi data akhir

In [8]:
connection.register('dataset_bersih', df_dedup)

validasi = sql("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(DISTINCT ticker) AS jumlah_ticker,
        MIN(tanggal) AS tanggal_awal,
        MAX(tanggal) AS tanggal_akhir,
        SUM(CASE WHEN open <= 0 OR high <= 0 OR low <= 0 OR close <= 0 THEN 1 ELSE 0 END) AS harga_nonpositif,
        SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS volume_negatif,
        SUM(CASE WHEN high < low THEN 1 ELSE 0 END) AS high_lt_low,
        SUM(CASE WHEN close > high OR close < low OR open > high OR open < low THEN 1 ELSE 0 END) AS ohlc_tidak_konsisten
    FROM dataset_bersih
""")
display(validasi)

kunci_ganda = sql("""
    SELECT COUNT(*) AS kunci_ganda FROM (
        SELECT ticker, tanggal FROM dataset_bersih GROUP BY ticker, tanggal HAVING COUNT(*) > 1
    )
""")
display(kunci_ganda)

per_ticker = sql("""
    SELECT ticker, COUNT(*) AS jumlah_baris, MIN(tanggal) AS awal, MAX(tanggal) AS akhir
    FROM dataset_bersih GROUP BY ticker ORDER BY jumlah_baris
""")
print('Ticker dengan data paling sedikit:')
display(per_ticker.head(10))
print('Ticker dengan data paling banyak:')
display(per_ticker.tail(5))

total_baris,jumlah_ticker,tanggal_awal,tanggal_akhir,harga_nonpositif,volume_negatif,high_lt_low,ohlc_tidak_konsisten
i64,i64,date,date,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
965539,318,1990-05-08,2025-07-04,0,0,0,0


kunci_ganda
i64
0


Ticker dengan data paling sedikit:


ticker,jumlah_baris,awal,akhir
str,i64,date,date
"""BPII""",656,2014-07-11,2025-07-04
"""BSWD""",714,2005-06-30,2025-07-02
"""HDTX""",831,2005-06-30,2019-05-28
"""BCIC""",847,2020-07-28,2025-01-30
"""GOLL""",963,2014-12-23,2019-01-29
"""CTBN""",1139,2005-07-04,2025-07-04
"""JSPT""",1142,2005-07-04,2025-07-04
"""DEFI""",1195,2005-06-30,2025-07-01
"""IBST""",1247,2012-09-03,2025-07-03


Ticker dengan data paling banyak:


ticker,jumlah_baris,awal,akhir
str,i64,date,date
"""AKRA""",5855,1994-10-03,2025-07-04
"""BBNI""",6037,1996-11-26,2025-07-04
"""ASII""",6770,1993-09-16,2025-07-04
"""JPFA""",7379,1990-05-09,2025-07-04
"""GJTL""",8221,1990-05-08,2025-07-04


8. Simpan data bersih

In [9]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / 'stocks_bersih.csv'
OUTPUT_PARQUET = OUTPUT_DIR / 'stocks_bersih.parquet'

# CSV: pemisah koma standar, titik sebagai desimal, tanggal format YYYY-MM-DD
df_dedup.write_csv(OUTPUT_CSV)

# Opsional: Parquet lebih kecil dan lebih cepat dibaca ulang
df_dedup.write_parquet(OUTPUT_PARQUET)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'CSV tersimpan di: {OUTPUT_CSV}')
print(f'Parquet tersimpan di: {OUTPUT_PARQUET}')

Baris sebelum cleaning: 1,048,575
Baris sesudah cleaning: 965,539
CSV tersimpan di: /home/jovyan/work/data/processed/stocks_bersih.csv
Parquet tersimpan di: /home/jovyan/work/data/processed/stocks_bersih.parquet


9. Log perubahan data

In [10]:
log = pl.DataFrame({
    'tahap': [
        'Data awal',
        'Missing values + gagal parse',
        'Logika harga OHLC / tanggal',
        'Outlier volume (IQR log10)',
        'Buang duplikat penuh',
        'Buang kunci (ticker, tanggal) ganda',
    ],
    'baris': [
        len(df), len(df_missing_handled), len(df_valid),
        len(df_outlier_handled), len(df_beda), len(df_dedup),
    ],
}).with_columns(
    (-pl.col('baris').diff()).alias('baris_dibuang')
).with_columns(
    (pl.col('baris_dibuang') / len(df) * 100).round(3).alias('persen_dari_awal')
)
display(log)

total_dibuang = len(df) - len(df_dedup)
print(f'Total dibuang: {total_dibuang:,} baris ({total_dibuang / len(df) * 100:.2f}% dari data awal)')

tahap,baris,baris_dibuang,persen_dari_awal
str,i64,i64,f64
"""Data awal""",1048575,null,null
"""Missing values + gagal parse""",1048575,0,0.0
"""Logika harga OHLC / tanggal""",977258,71317,6.801
"""Outlier volume (IQR log10)""",965539,11719,1.118
"""Buang duplikat penuh""",965539,0,0.0
"""Buang kunci (ticker, tanggal) …",965539,0,0.0


Total dibuang: 83,036 baris (7.92% dari data awal)


10. Ringkasan masalah kualitas data

In [11]:
n_missing_dibuang = len(df) - len(df_missing_handled)

masalah = pl.DataFrame({
    'dimensi_kualitas': [
        'Completeness / parsing (kosong atau gagal dibaca)',
        'Validity (logika harga OHLC, tanggal)',
        'Validity (outlier volume)',
        'Uniqueness (duplikat penuh)',
        'Uniqueness (kunci ticker+tanggal ganda)',
    ],
    'jumlah_baris': [n_missing_dibuang, n_logika_dibuang, n_outlier, n_duplikat_penuh, n_konflik_dibuang],
}).with_columns(
    (pl.col('jumlah_baris') / len(df) * 100).round(3).alias('persen_dari_awal')
).sort('jumlah_baris', descending=True)
display(masalah)

terbesar = masalah.row(0, named=True)

ringkasan = f"""
## Ringkasan Data Cleaning Saham IDX

**Dimensi kualitas paling bermasalah:** {terbesar['dimensi_kualitas']},
sebanyak {terbesar['jumlah_baris']:,} baris ({terbesar['persen_dari_awal']}% dari data awal).

**Parsing & missing values:** file dibaca dengan pemisah `;`, semua kolom sebagai teks lalu
dikonversi. Format angka Indonesia (koma desimal) dan notasi ilmiah ditangani. Prefix `IDX:` dihapus
dari simbol. Baris dengan nilai kosong atau gagal parse dibuang ({n_missing_dibuang:,} baris).
Imputasi tidak dipakai karena harga saham tidak boleh dikarang.

**Logika harga:** baris dengan harga <= 0, volume negatif, high < low, atau open/close di luar
[low, high] dibuang ({n_logika_dibuang:,} baris). Baris dengan rasio max/min OHLC > 5
({len(skala_aneh):,} baris) hanya dilaporkan sebagai tanda skala angka yang tidak konsisten.

**Outlier:** IQR faktor {FAKTOR_IQR} pada log10(volume), {n_outlier:,} baris dibuang.
{len(lompatan):,} lonjakan harga antar hari hanya dilaporkan (bisa berupa stock split yang sah).

**Deduplikasi:** {n_duplikat_penuh:,} duplikat penuh dan {n_konflik_dibuang:,} baris dengan kunci
(ticker, tanggal) ganda dibuang. Untuk kunci ganda, baris terakhir yang dipertahankan.

**Jumlah baris:** dari {len(df):,} menjadi {len(df_dedup):,}
(berkurang {total_dibuang:,} atau {total_dibuang / len(df) * 100:.2f}%).

**Sumber data:** Indonesia Stock Market (IDX) Price Data, Kaggle (bestagi).
"""
print(ringkasan)


dimensi_kualitas,jumlah_baris,persen_dari_awal
str,i64,f64
"""Validity (logika harga OHLC, t…",71317,6.801
"""Validity (outlier volume)""",11719,1.118
"""Completeness / parsing (kosong…",0,0.0
"""Uniqueness (duplikat penuh)""",0,0.0
"""Uniqueness (kunci ticker+tangg…",0,0.0



## Ringkasan Data Cleaning Saham IDX

**Dimensi kualitas paling bermasalah:** Validity (logika harga OHLC, tanggal),
sebanyak 71,317 baris (6.801% dari data awal).

**Parsing & missing values:** file dibaca dengan pemisah `;`, semua kolom sebagai teks lalu
dikonversi. Format angka Indonesia (koma desimal) dan notasi ilmiah ditangani. Prefix `IDX:` dihapus
dari simbol. Baris dengan nilai kosong atau gagal parse dibuang (0 baris).
Imputasi tidak dipakai karena harga saham tidak boleh dikarang.

**Logika harga:** baris dengan harga <= 0, volume negatif, high < low, atau open/close di luar
[low, high] dibuang (71,317 baris). Baris dengan rasio max/min OHLC > 5
(35,379 baris) hanya dilaporkan sebagai tanda skala angka yang tidak konsisten.

**Outlier:** IQR faktor 3.0 pada log10(volume), 11,719 baris dibuang.
27,519 lonjakan harga antar hari hanya dilaporkan (bisa berupa stock split yang sah).

**Deduplikasi:** 0 duplikat penuh dan 0 baris dengan kunci
(ticker, tanggal) ganda dibuang. Untu